In [ ]:
'''
Look at each beam trigger.
Find adjacent channel with a signal 
Plot it 
'''


In [ ]:
%load_ext autoreload
%autoreload 2

import importlib
import waffles.np04_analysis.lightyield_vs_energy.scripts.utils as utils_module
from waffles.np04_analysis.lightyield_vs_energy.scripts.utils import *
import ast


In [ ]:
energy_dict =  {1: 0, 2: 118, 3: 119, 5: 175, 7: 253} # Crossing point between langauss and gaussian distribution - to use for beam event selection trigger 

energies = [1,2,3,5,7] #[1,2,3,5,7]

apa_studied = ['1'] #['1', '2']

input_folder = f"/afs/cern.ch/work/a/anbalbon/private/waffles/src/waffles/np04_analysis/lightyield_vs_energy/output"
output_folder= f"{input_folder}/adjacent_chanels_study"


In [ ]:
adjacent_channels_apa1_map , adjacent_channels_apa2_map, _ = adjacent_channel_info()

adiacent_channels_map = {}
if '1' in apa_studied:
    adiacent_channels_map['1'] = adjacent_channels_apa1_map
if '2' in apa_studied:
    adiacent_channels_map['2'] = adjacent_channels_apa2_map


adiacent_channels_map = {'1': [[{'apa' : 1, 'end': 104, 'ch': 15},{'apa' : 1, 'end': 104, 'ch': 12}]]}

In [ ]:
ENERGY_adiacent_channels_list = {}
mean_pe = []
mean_pe_err = []
std_pe = []

for energy in energies: 
    apa1_mean_pe_threshold = energy_dict[energy]

    apa12_energy_folder = f"{input_folder}/apa1_vs_apa2/{energy}GeV"

    merged_dict = load_energy_json_dict(energy, apa12_energy_folder, output_folder)


    # SELECTING TRIGGERS INDEXES WHERE APA1 MEAN PE > THRESHOLD
    dic_trigger_index = {}
    for _to_, to_dict in merged_dict.items(): # looking at each "0_to_10", "10_to_20", ...
        index_list = []
        for i, trigger_mean_pe in enumerate(to_dict['1']['mean']): # looking at each trigger
            try:
                if trigger_mean_pe > apa1_mean_pe_threshold: # checking if the mean photoelectrons in APA1 is above threshold
                    index_list.append(i)
            except:
                continue
        dic_trigger_index[_to_] = index_list # saving the list of trigger indexes that passed the threshold for this "_to_"


    adiacent_channels_info_dict =[] 
    adiacent_channels_list = {'A':[], 'eA': [], 'B':[], 'eB':[]}
   # {'ch1': {'apa' : , 'end': , 'ch': , 'n_pe': , 'e_n_pe': }, 'ch2': {'apa' : , 'end': , 'ch': , 'n_pe': , 'e_n_pe': }, 'trigger' : }
    
    # Searching for adjacent channel fired during the same beam trigger
    i=0
    n_pe_list = []
    for _to_, index_list in dic_trigger_index.items(): # looking at each "0_to_10", "10_to_20", ...
      
        for index in index_list: # Aceessing beam trigger information
            for apa, adiacent_channels_apa in adiacent_channels_map.items(): # looking at APA1 and APA2
                for end, end_dic in merged_dict[_to_][str(apa)]["channel_dic"][index].items(): # looking at each endpoint
                    
                    # for ch, ch_dic in end_dic.items():
                    #     n_pe_list.append(ch_dic['n_pe'])
                    
                    ch_list = list(end_dic.keys())
                    for ch_pairs in adiacent_channels_apa:
                        apa1 = str(ch_pairs[0]['apa'])
                        end1 = str(ch_pairs[0]['end'])
                        ch1 = str(ch_pairs[0]['ch'])
                       
                        apa2 = str(ch_pairs[1]['apa'])
                        end2 = str(ch_pairs[1]['end'])
                        ch2 = str(ch_pairs[1]['ch'])

                       
                        if ((apa1 == apa) and (end1 == end) and (ch1 in ch_list)) and ((apa2 == apa) and (end2 == end) and (ch2 in ch_list)):
                            n_pe1 = merged_dict[_to_][apa1]["channel_dic"][index][end1][ch1]['n_pe']
                            e_n_pe1 = merged_dict[_to_][apa1]["channel_dic"][index][end1][ch1]['e_n_pe']
                            n_pe2 = merged_dict[_to_][apa2]["channel_dic"][index][end2][ch2]['n_pe']
                            e_n_pe2 = merged_dict[_to_][apa2]["channel_dic"][index][end2][ch2]['e_n_pe']
                            adiacent_channels_info_dict.append({'ch1': {'apa' : apa1, 'end': end1, 'ch': ch1, 'n_pe': n_pe1, 'e_n_pe': e_n_pe1}, 'ch2': {'apa' : apa2, 'end': end2, 'ch': ch2, 'n_pe': n_pe2, 'e_n_pe': e_n_pe2}, 'trigger' : merged_dict[_to_]['trigger time'][index]})
                        

                            adiacent_channels_list['A'].append(n_pe1) 
                            adiacent_channels_list['eA'].append(e_n_pe1) 
                            adiacent_channels_list['B'].append(n_pe2) 
                            adiacent_channels_list['eB'].append(e_n_pe2) 

                            n_pe_list.append(n_pe1)
                            # n_pe_list.append(n_pe2)
                        
                        else: 
                            i+=1

    mean_pe.append(np.mean(n_pe_list))  
    mean_pe_err.append(np.mean(n_pe_list)/np.sqrt(len(n_pe_list)))
    std_pe.append(np.std(n_pe_list))                      
    ENERGY_adiacent_channels_list[energy] = adiacent_channels_list

mean_pe = np.array(mean_pe)
mean_pe_err =  np.array(mean_pe_err)
std_pe = np.array(std_pe)



In [ ]:

# Color map by energy
colors = {1: "red", 2: "green", 3: "blue", 5: "orange", 7: "purple"}

energies = list(ENERGY_adiacent_channels_list.keys())
n = len(energies)

# Residual definitions to be shown in rows 2–4
strategies = [
    'Residual definition',
    'Distance from y=x',
    'Distance from y=A+Bx',
    'N1-N2',
    '(N1-N2)/((N1+N2)/2)'
]

# ===== 4 rows: 1 scatter + 3 residual plots =====
fig, axes = plt.subplots(len(strategies)+1, n, figsize=(5*n, 20))
fig.subplots_adjust(hspace=0.3, top=0.92, bottom=0.05)

# =========================================================
# ===== ROW 1 — SCATTER + ODR FIT
# =========================================================
fit_results = {}  # store fit parameters for residuals

# mean_pe = []
# mean_pe_err = []
# std_pe = []

for j, energy in enumerate(energies):

    ax = axes[0, j]
    dict_ = ENERGY_adiacent_channels_list[energy]

    x  = np.asarray(dict_['A'],  dtype=float)
    ex = np.asarray(dict_['eA'], dtype=float)
    y  = np.asarray(dict_['B'],  dtype=float)
    ey = np.asarray(dict_['eB'], dtype=float)

    # Data with errors
    ax.errorbar(
        x, y,
        xerr=ex, yerr=ey,
        fmt='o',
        markersize=5,
        capsize=3,
        elinewidth=1,
        markeredgecolor='black',
        markerfacecolor=colors[energy],
        label="Data"
    )

    xmin, xmax = min(x), max(x)

    # Reference line y = x
    ax.plot([xmin, xmax], [xmin, xmax], 'k--', label='Expected y = x')

    # ----- ODR linear fit -----
    data = RealData(x, y, sx=ex, sy=ey)
    model = Model(linear_array)
    odr = ODR(data, model, beta0=curve_fit(linear, x, y, sigma=ey, absolute_sigma=True)[0])
    out = odr.run()

    A, B = out.beta
    eA, eB = out.sd_beta

    fit_results[energy] = (A, B, eA, eB)

    y_fit = linear(x, A, B)

    r_squared = r2_score(y, y_fit)
    chi2rid = out.sum_square / (len(x) - len(out.beta))

    label_fit = (
        f'y = A + Bx\n'
        f'A = ({fmt(A, eA)})\n'
        f'B = ({fmt(B, eB)})\n'
        f'$R^2$ = {r_squared:.3f}\n'
        f'$\chi^2_{{rid}}$ = {chi2rid:.2f}'
    )

    ax.plot([xmin, xmax],
            linear_array([A,B], np.array([xmin, xmax])),
            'r-', label=label_fit)

    ax.set_xlabel(r"$N_{PE}$ — ch A")
    ax.set_ylabel(r"$N_{PE}$ — ch B")
    ax.set_title(f"{energy} GeV")
    ax.grid(True)
    ax.legend(loc='upper left', fontsize=8)

    # xy = np.concatenate([x, y])
    # mean_pe.append(np.mean(xy))
    # mean_pe_err.append(np.mean(xy)/np.sqrt(len(xy)))
    # std_pe.append(np.std(xy, ddof=1))

# mean_pe = np.array(mean_pe)
# mean_pe_err =  np.array(mean_pe_err)
# std_pe = np.array(std_pe)

# =========================================================
# ===== ROWS 2–4 — RESIDUAL HISTOGRAMS + GAUSSIAN FIT
# =========================================================

all_mu       = {s: [] for s in strategies}
all_mu_err   = {s: [] for s in strategies}
all_sigma        = {s: [] for s in strategies}
all_sigma_err    = {s: [] for s in strategies}
all_mean = {s: [] for s in strategies}
all_std = {s: [] for s in strategies}
all_mean_W     = {s: [] for s in strategies}
all_mean_err_W = {s: [] for s in strategies}

energy_vals_global = []

for i, residual_strategy in enumerate(strategies):

    energy_vals = []
    mu_list, mu_err_list = [], []
    sigma_list, sigma_err_list = [], []
    mean_list = []
    stds_list = []
    w_mean_list, w_mean_err_list = [], []

    for j, energy in enumerate(energies):

        ax = axes[i+1, j]
        dict_ = ENERGY_adiacent_channels_list[energy]

        x  = np.asarray(dict_['A'],  dtype=float)
        ex = np.asarray(dict_['eA'], dtype=float)
        y  = np.asarray(dict_['B'],  dtype=float)
        ey = np.asarray(dict_['eB'], dtype=float)

        if x.size == 0 or y.size == 0:
            ax.set_visible(False)
            continue

        A, B, eA, eB = fit_results[energy]

        # ----- Compute residuals -----
        if residual_strategy == 'Residual definition':
            y_model = A + B*x
            sigma_tot = np.sqrt(ey**2 + (B*ex)**2)
            residuals = (y - y_model) / sigma_tot
            residuals_err = []

        elif residual_strategy == 'Distance from y=x':
            residuals = (y - x) / np.sqrt(2)
            residuals_err = np.sqrt(ey**2 + ex**2) / np.sqrt(2)

        elif residual_strategy == 'Distance from y=A+Bx':
            den = np.sqrt(B**2 + 1)
            num = B*x - y + A
            residuals = num / den

            dr_dx = B / den
            dr_dy = -1 / den
            dr_dA = 1 / den
            dr_dB = (x*den - num*(B/den)) / (den**2)

            residuals_err = np.sqrt(
                (dr_dx * ex)**2 +
                (dr_dy * ey)**2 +
                (dr_dA * eA)**2 +
                (dr_dB * eB)**2
            )
        elif residual_strategy =='N1-N2':
            residuals =  x-y 
            residuals_err = np.sqrt(ey**2 + ex**2)
        
        elif residual_strategy == '(N1-N2)/((N1+N2)/2)':
            residuals = (x-y)/((x+y)/2)
            residuals_err = []

        else:
            continue

        # ----- Basic statistics -----
        N = len(residuals)
        mean_0 = np.mean(residuals)
        std_0  = np.std(residuals, ddof=1)

        if len(residuals_err) > 0:
            weights = 1 / residuals_err**2
            w_mean = np.average(residuals, weights=weights)
            w_mean_err = np.sqrt(1 / np.sum(weights))
            label = f"N = {N}\n$\mu$ = {mean_0:.2}\n$\sigma$ = {std_0:.2f}"#\n$\mu_{{W}}$ = {fmt(w_mean, w_mean_err)}"
        else:
            w_mean = np.nan
            w_mean_err = np.nan
            label = f"N = {N}\n$\mu$ = {mean_0:.4f}\n$\sigma$ = {std_0:.4f}"

        # ----- Histogram -----
        Range = [np.percentile(residuals, 0.3), np.percentile(residuals, 99.8)]
        bins = 50

        ax.hist(
            residuals,
            bins=bins,
            range=Range,
            color=colors[energy],
            alpha=0.8,
            label=label
        )

        # ===== Gaussian fit of histogram =====
        counts, bin_edges = np.histogram(residuals, bins=bins, range=Range)

        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
        bin_width = bin_edges[1] - bin_edges[0]

        mask = counts > 0
        x_fit = bin_centers[mask]
        y_fit = counts[mask]

        sx = np.full_like(x_fit, bin_width / 2)
        sy = np.sqrt(y_fit)

        data_g = RealData(x_fit, y_fit, sx=sx, sy=sy)
        model_g = Model(gaussian_array)

        odr_g = ODR(data_g, model_g, beta0=[mean_0, std_0, np.max(y_fit)])
        out_g = odr_g.run()

        mu, sigma, A_g = out_g.beta
        emu, esigma, eA_g = out_g.sd_beta

        y_model_g = gaussian(x_fit, mu, sigma, A_g)
        r_squared = r2_score(y_fit, y_model_g)
        chi2rid = out_g.sum_square / (len(x_fit)-len(out.beta))
        # dfdx = y_model_g * (-(x_fit - mu) / sigma**2)
        # sigma_tot = np.sqrt(sy**2 + (dfdx * sx)**2)
        # chi2_val, chi2rid, ndf = chi2_func(y_fit, y_model_g, sigma_tot, len(out_g.beta))

        # ----- Plot Gaussian fit -----
        label_fit = (
            f"$\mu$ = {fmt(mu, emu)}\n"
            f"$\sigma$ = {fmt(sigma, esigma)}\n"
            f"A = {fmt(A_g,eA_g)}\n"
            f"R$^2$ = {r_squared:.3f}\n"
            f"$\chi^2_{{rid}}$ = {chi2rid:.3f}"
        )

        x_plot = np.linspace(Range[0], Range[1], 500)
        y_plot = gaussian(x_plot, mu, sigma, A_g)

        ax.plot(x_plot, y_plot, 'k-', linewidth=2, label=label_fit)
        ax.axvline(mu, linestyle='--')

        ax.set_title(f"{energy} GeV — {residual_strategy}", fontsize=10)
        ax.set_ylabel("Counts", fontsize=9)
        ax.set_xlabel("Residuals", fontsize=9)

        ax.legend(loc="upper left", fontsize=8)
        ax.grid(True)

        # Store fit results
        energy_vals.append(float(energy))
        mu_list.append(mu)
        mu_err_list.append(emu)
        sigma_list.append(sigma)
        sigma_err_list.append(esigma)
        mean_list.append(mean_0)
        stds_list.append(std_0)
        w_mean_list.append(w_mean)
        w_mean_err_list.append(w_mean_err)


    if not energy_vals_global:
        energy_vals_global = energy_vals.copy()

    all_mu[residual_strategy]       = np.array(mu_list)
    all_mu_err[residual_strategy]   = np.array(mu_err_list)
    all_sigma[residual_strategy]        = np.array(sigma_list)
    all_sigma_err[residual_strategy]    = np.array(sigma_err_list)
    all_mean[residual_strategy] = np.array(mean_list)
    all_std[residual_strategy] = np.array(stds_list)
    all_mean_W[residual_strategy]     = np.array(w_mean_list)
    all_mean_err_W[residual_strategy] = np.array(w_mean_err_list)


# =========================================================
# ===== GLOBAL LABELS
# =========================================================
fig.suptitle(
    "Adjacent channel in the same beam trigger",
    fontsize=25
)

fig.text(
    0.85, 0.96,
    r'$\bf{ProtoDUNE\!-\!HD}$ Preliminary',
    ha='center', va='center',
    fontsize=11
)

plt.savefig(
    f"{output_folder}/adjacent_channels_same_trigger.png",
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
# ==========================================
# 🔵 FIGURA RIASSUNTIVA: Mean + Std + Weighted Mean
# ==========================================

fig, axes = plt.subplots(1, 3, figsize=(18,10), sharex=True)
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

# ======================
# 1️⃣ Mean (non pesata)
# ======================
for i, strategy in enumerate(strategies):
    color = colors[i % len(colors)]
    axes[0].errorbar(
        energy_vals_global,
        all_mu[strategy],
        xerr = np.array(energy_vals_global)*0.05,
        yerr=all_mu_err[strategy],
        color = color, 
        marker='o',
        linestyle='-',
        markersize=4,
        linewidth=1,
        capsize=3,
        label=f"{strategy} - $\mu$ fit"
    )

axes[0].set_xlabel(r"$\langle E_{beam} \rangle$ [GeV]")
axes[0].set_ylabel("Mean of residuals [$N_{{PE}}$]")
axes[0].set_title("Residual Mean vs Energy", fontsize=13)
axes[0].grid(True)
axes[0].set_xlim(0, 8)
# axes[0].set_ylim(-8, 20)
axes[0].legend()


# ======================
# 2 Std
# ======================
for i, strategy in enumerate(strategies):
    color = colors[i % len(colors)]
    axes[1].errorbar(
        energy_vals_global,
        all_sigma[strategy],
        xerr = np.array(energy_vals_global)*0.05,
        yerr=all_sigma_err[strategy],
        color = color,
        marker='o',
        linestyle='-',
        markersize=4,
        linewidth=1,
        label=strategy
    )

axes[1].set_xlabel(r"$\langle E_{beam} \rangle$ [GeV]")
axes[1].set_ylabel("Std Dev of residuals [$N_{{PE}}$]")
axes[1].set_title("Residual Std Dev vs Energy", fontsize=13)
axes[1].grid(True)
axes[1].set_xlim(0, 8)
axes[1].legend()

# ======================
# 3 energy resolution
# ======================
for i, strategy in enumerate(strategies): #['Distance from y=x','Distance from y=A+Bx']:
    color = colors[i % len(colors)]
    x = np.array(energy_vals_global)
    ex = x*0.05

    ## dividing by energy
    # y = all_sigma[strategy]/x
    # ey = np.abs(y*np.sqrt((all_sigma_err[strategy]/all_sigma[strategy])**2+(ex/x)**2))
    # y_label = "$\sigma$/$\langle E_{beam} \rangle$ [$N_{{PE}}$/GeV]"

    ## dividing by mean Npe 1
    y = np.abs(all_sigma[strategy]/mean_pe)
    ey = np.abs(y*np.sqrt((all_sigma_err[strategy]/all_sigma[strategy])**2+(mean_pe_err/mean_pe)**2))
    y_label = r"$\frac{\sigma}{\langle N_{PE} \rangle}\;[N_{PE}/\mathrm{GeV}]$"


    axes[2].errorbar(
    x, y, xerr=ex, yerr=ey,
    marker='o',
    linestyle='',
    markersize=4,
    linewidth=1,
    color = color
    #label='strategy'
    )


    data = RealData(x, y, sx=ex, sy=ey)
    model = Model(energy_resolution_fit_array)

    odr = ODR(data, model, beta0=[0.02, 0.3, 1.0])
    out = odr.run()

    p0, p1, p2 = out.beta
    ep0, ep1, ep2 = out.sd_beta

    chi2_red = out.sum_square / (len(x) - len(out.beta))

    y_fit_label = (f"Fit - {strategy}\n"
    f"p0 = {fmt(p0,ep0)}\n"
    f"p1 = {fmt(p1,ep1)}\n"
    f"p2 = {fmt(p2,ep2)}\n"
    rf"$\chi^2_{{rid}}$ = {chi2_red:.3f}")

    x_plot = np.linspace(min(x), max(x), 500)
    y_plot = energy_resolution_fit(x_plot, p0, p1, p2)
    plt.plot(x_plot, y_plot, color = color, label=y_fit_label)


axes[2].set_xlabel(r"$\langle E_{beam} \rangle$ [GeV]")
axes[2].set_ylabel(y_label)
axes[2].set_title("Energy resolution", fontsize=13)
axes[2].grid(True)
axes[2].set_xlim(0, 8)
# axes[2].set_ylim(0, 100)
axes[2].legend()

fig.text(1, 1, r'$\bf{ProtoDUNE\!-\!HD}$ Preliminary',
         fontsize=11, ha='right', va='top')


plt.suptitle(f"APA 12", fontsize=15)
plt.tight_layout()

plt.savefig(
    f"{output_folder}/adjacent_channels_same_trigger_summary.png",
    dpi=300
)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18,10), sharex=True)

i=0

for strategy in ['N1-N2', '(N1-N2)/((N1+N2)/2)']:
    x = np.array(energy_vals_global)
    ex = x*0.05

    if strategy == 'N1-N2':
        y = (1/np.sqrt(2))* (all_sigma[strategy]/mean_pe)
        ey = y * np.sqrt((all_sigma_err[strategy]/all_sigma[strategy])**2+(mean_pe_err/mean_pe)**2)
        y_label = r'$\frac{1}{\sqrt{2}} \cdot \sigma(N_1 - N_2) \cdot \frac{1}{\langle N_1 \rangle}$'
    elif strategy == '(N1-N2)/((N1+N2)/2)':
        y = (1/np.sqrt(2))* all_sigma[strategy]
        ey = (1/np.sqrt(2))* all_sigma_err[strategy]
        y_label = r'$\frac{1}{\sqrt{2}} \cdot \sigma\!\left(\frac{N_1 - N_2}{(N_1 + N_2) /2}\right)$'
    else:
        continue


    axes[i].errorbar(x,
            y,
            xerr = ex,
            yerr = ey,
            marker='o',
            linestyle='',
            markersize=4,
            linewidth=1,
            capsize=3,
            label=f"Data"
        )


    for fit_func_array, fit_func, fit_func_label in zip([energy_resolution_fit_array], [energy_resolution_fit], [r'$y=\sqrt{p_0^2 + \left(\frac{p_1}{\sqrt{x}}\right)^2 + \left(\frac{p_2}{x}\right)^2}$']): #zip([energy_resolution_fit_array_1, energy_resolution_fit_array_2], [energy_resolution_fit_1, energy_resolution_fit_2], [r'$y=\sqrt{p_0^2 + \left(\frac{p_1}{\sqrt{x}}\right)^2 + \left(\frac{p_2}{x}\right)^2}$', r'y=$p_0 + \frac{p_1}{\sqrt{x}} + \frac{p_2}{x}$']):

        p0_init = np.min(y)                  
        p1_init = np.sqrt(max(y[0]**2 - p0_init**2, 1e-6)) * np.sqrt(x[0])
        p2_init = 0.1                       
        
        data = RealData(x, y, sx=ex, sy=ey)
        model = Model(fit_func_array)

        odr = ODR(data, model, beta0=np.array([p0_init, p1_init, p2_init]))
        odr.set_job(fit_type=0)
        out = odr.run()

        p0, p1, p2 = out.beta
        ep0, ep1, ep2 = out.sd_beta

        y_fit = fit_func(x,p0,p1,p2)
        # chi2_red = out.sum_square / (len(x) - len(out.beta)) # --> SBAGLIATO, non normalizza per sigma
        dfdx = -(p1**2/x**2 + 2*p2**2/x**3) / (2*y_fit)
        sigma_tot = np.sqrt(ey**2 + (dfdx * ex)**2)
        chi2_val, chi2_red, ndf = chi2_func(y, y_fit, sigma_tot, len(out.beta))

        r_squared = r2_score(y,y_fit)
   
        y_fit_label = (f"Fit {fit_func_label}\n"
        f"p0 = {fmt(p0,ep0)}\n"
        f"p1 = {fmt(p1,ep1)}\n"
        f"p2 = {fmt(p2,ep2)}\n"
        f"$R^2$ = {r_squared:.3f}\n"
        r"$\chi^2_{rid}$" +f" = {chi2_red:.3f}")

        x_plot = np.linspace(min(x), max(x), 500)
        y_plot = fit_func(x_plot, p0, p1, p2)
        axes[i].plot(x_plot, y_plot, label=y_fit_label)

    axes[i].set_xlabel(r"$\langle E_{beam} \rangle$ [GeV]")
    axes[i].set_ylabel(y_label)
    axes[i].set_title(f"{strategy}", fontsize=13)
    axes[i].grid(True)
    axes[i].set_xlim(0, 8)
    axes[i].legend()
    i+=1

fig.text(1, 1, r'$\bf{ProtoDUNE\!-\!HD}$ Preliminary',
         fontsize=11, ha='right', va='top')

plt.suptitle(f"New energy resolution computation", fontsize=15)
plt.tight_layout()

plt.savefig(
    f"{output_folder}/adjacent_channels_same_trigger_summary_NEW.png",
    dpi=300
)
plt.show()